In this notebook we will implement all changes to the data discovered during EDA.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [2]:
train = pd.read_csv('data/train.csv')
score = pd.read_csv('data/score.csv')

# Data prep

## Converitng data types

In [3]:
# convert date columns to datetime format
date_columns = ["snapshot_date", "policy_effective_date", "policy_expiration_date", "first_claim_reported_date"]

for col in date_columns:
    train[col] = pd.to_datetime(train[col], errors='coerce')
    score[col] = pd.to_datetime(score[col], errors='coerce')

# extract policy year for train to use later train/test split
train['policy_year'] = train['policy_effective_date'].dt.year

In [4]:
# clean text columns by stripping whitespace and converting to lowercase
text_columns = ["business_type", "state", "payment_frequency"]

for col in text_columns:
    train[col] = train[col].str.strip().str.lower()
    score[col] = score[col].str.strip().str.lower()

## Missing values

In [5]:
# fill missing values in risk_score_external with average by business_type and coverage_type
def impute_risk_score(df: pd.DataFrame) -> pd.DataFrame:
    df_cleaned = df.copy()
    
    group_means = df_cleaned.groupby(['business_type', 'coverage_type'])['risk_score_external'].transform('mean')
    
    df_cleaned['risk_score_external'] = df_cleaned['risk_score_external'].fillna(group_means)
    
    global_mean = df_cleaned['risk_score_external'].mean()
    df_cleaned['risk_score_external'] = df_cleaned['risk_score_external'].fillna(global_mean)
    
    return df_cleaned

In [6]:
train = impute_risk_score(train)
score = impute_risk_score(score)

In [7]:
# fill missing values for numeric columns with 0
train[train.select_dtypes("number").columns] = train.select_dtypes("number").fillna(0)
score[score.select_dtypes("number").columns] = score.select_dtypes("number").fillna(0)

In [8]:
# fill missing values for payment_frequency columns with 'missing'
train['payment_frequency'] = train['payment_frequency'].fillna('missing')
score['payment_frequency'] = score['payment_frequency'].fillna('missing')

## Prepare target variable

In [9]:
# ---- target ----
# using binary had-a-claim flag as target
train["target"] = (train["claim_count"] > 0).astype(int)

## Data split

In [10]:
# split the dataset based on the coverage
train_al = train[train['coverage_type'] == 'AL'].copy()
train_apd = train[train['coverage_type'] == 'APD'].copy()

score_al = score[score['coverage_type'] == 'AL'].copy()
score_apd = score[score['coverage_type'] == 'APD'].copy()

## Handle outliers

In [11]:
# delete rows with vehicle_count = 0
train_apd = train_apd[train_apd['vehicle_count'] > 0].copy()

score_apd = score_apd[score_apd['vehicle_count'] > 0].copy()

In [12]:
# filtering out rows with annual premium less than or equal to 0
train_al = train_al[train_al['annual_premium'] > 0].copy()
train_apd = train_apd[train_apd['annual_premium'] > 0].copy()

score_al = score_al[score_al['annual_premium'] > 0].copy()
score_apd = score_apd[score_apd['annual_premium'] > 0].copy()

# Modeling

## Encoding

## Feature selection

Select features for both subset based on EDA.

In [13]:
feat_cols_al = ["vehicle_count", "num_heavy_vehicles", "risk_score_external", "prior_year_mileage_000", "prior_al_claim_count",
                "driver_count", "vehicle_avg_age", "late_payment_count", "coverage_limit_000", "prior_loss_amount", 
                "business_type", "payment_frequency", "state", "deductible"]

feat_cols_apd = ["vehicle_count", "num_heavy_vehicles", "risk_score_external", "prior_year_mileage_000", "prior_apd_claim_count",
                "driver_count", "vehicle_avg_age", "late_payment_count", "coverage_limit_000", "prior_loss_amount", "annual_premium",
                "years_in_business",
                "business_type", "payment_frequency", "state", "deductible"]

In [14]:
X_al = train_al[feat_cols_al + ["policy_year"]]
y_al = train_al[["target", "policy_year"]]

X_apd = train_apd[feat_cols_apd + ["policy_year"]]
y_apd = train_apd[["target", "policy_year"]]

In [15]:
X_al_score = score_al[feat_cols_al]
X_apd_score = score_apd[feat_cols_apd]

## Train, test split

In [16]:
X_al_train = X_al[X_al['policy_year'].isin([2019, 2020, 2021])].drop(columns=['policy_year']).copy()
X_al_test = X_al[X_al['policy_year'].isin([2022])].drop(columns=['policy_year']).copy()

X_apd_train = X_apd[X_apd['policy_year'].isin([2019, 2020, 2021])].drop(columns=['policy_year']).copy()
X_apd_test = X_apd[X_apd['policy_year'].isin([2022])].drop(columns=['policy_year']).copy()

y_al_train = y_al[y_al['policy_year'].isin([2019, 2020, 2021])].drop(columns=['policy_year']).copy()
y_al_test = y_al[y_al['policy_year'].isin([2022])].drop(columns=['policy_year']).copy()

y_apd_train = y_apd[y_apd['policy_year'].isin([2019, 2020, 2021])].drop(columns=['policy_year']).copy()
y_apd_test = y_apd[y_apd['policy_year'].isin([2022])].drop(columns=['policy_year']).copy()

## Modeling pipeline

Becuase of business types mismatch between training and scoring dataset encoding, standardization, model fit need to be done in pipeline that will handle this.

In [21]:
num_cols = [col for col in train.select_dtypes(include=[np.number]).columns if col not in ["target", "claim_count", "has_safety_program"]]
num_cols_al = [col for col in num_cols if col in X_al_train.columns]
num_cols_apd = [col for col in num_cols if col in X_apd_train.columns]

skewed_cols = ['prior_loss_amount', 'annual_premium', 'prior_year_mileage_000', 'vehicle_count', 'driver_count']
skewed_cols_al = [col for col in skewed_cols if col in X_al_train.columns]
skewed_cols_apd = [col for col in skewed_cols if col in X_apd_train.columns]

low_cardinality_cols = ['business_type', 'payment_frequency']

high_cardinality_cols = ['state']

In [22]:
skewed_transformer = Pipeline(steps=[
    ('log1p', FunctionTransformer(np.log1p))
])

regular_transformer = RobustScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('num_log_scale', skewed_transformer, skewed_cols_al),
        ('num_regular', regular_transformer, num_cols_al),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_cardinality_cols),
        ('cat_te', TargetEncoder(smooth="auto"), high_cardinality_cols) 
    ]
)

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='saga'))
])

param_grid = {
        'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'classifier__l1_ratio': [0.2, 0.5, 0.8],  
        'classifier__class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=full_pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='roc_auc', 
    n_jobs=-1
)

grid_search.fit(X_al_train, y_al_train)

c:\Zurich\Case_study\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Zurich\Case_study\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...ver='saga'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.001, 0.01, ...], 'classifier__class_weight': [None, 'balanced'], 'classifier__l1_ratio': [0.2, 0.5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
